# YOLOv8 Dental Detection: Tooth Types + Damage

This notebook trains a YOLOv8 model to detect:
- **Tooth Types**: Incisors, Canines, Premolars, Molars
- **Damage**: Caries, Cavity, Decay, Crack
- **Position**: Upper/Lower (calculated from coordinates)

## Dataset Requirements

Your dataset must have these classes:
- `incisor`, `canine`, `premolar`, `molar` (tooth types)
- `caries`, `cavity`, `decay`, `crack` (damage types)

## Notebook Structure
1. Dataset Preparation
2. Model Training
3. Inference & Visualization
4. Model Evaluation
5. Application Integration

## Dataset Merger for YOLO Dataset

In [ ]:

"""
Merge multiple YOLO-format dental datasets into one unified dataset.

Usage:
1. Download datasets into separate folders
2. Update DATASET_SOURCES below
3. Run this script
"""

import shutil
import os
from pathlib import Path
import random

# ============================================
# CONFIGURATION
# ============================================
OUTPUT_DIR = Path("./yolo_dataset_merged")
TRAIN_RATIO = 0.8
VAL_RATIO = 0.15
TEST_RATIO = 0.05

# Unified class mapping: map source class names -> unified IDs
# Update these based on the datasets you download
UNIFIED_CLASSES = {
    # Tooth types (if available, map to generic 'tooth' for now)
    'tooth':      0,
    'Tooth':      0,
    'teeth':      0,
    'incisor':    0,   # Merge into 'tooth' unless you have enough data per type
    'canine':     0,
    'premolar':   0,
    'molar':      0,

    # Damage types
    'caries':     1,
    'Caries':     1,
    'carie':      1,
    'decay':      1,    # Map decay -> caries (same condition)
    'Decay':      1,

    'cavity':     2,
    'Cavity':     2,

    'crack':      3,
    'Crack':      3,
    'fracture':   3,   # Fracture = crack
}

UNIFIED_NAMES = ['tooth', 'caries', 'cavity', 'crack']

# List of dataset directories (each must have train/images, train/labels, etc.)
DATASET_SOURCES = [
    {
        "path": "./yolo_dataset",           # Your existing converted dataset
        "class_map": None,                   # None = already uses unified IDs
    },
    {
        "path": "./extra_dataset_1",         # Additional dataset
        "class_map": {                       # Map: source_class_id -> source_class_name
            0: "tooth",                      # Then we look up in UNIFIED_CLASSES
            1: "caries",
            2: "cavity",
            3: "crack",
        },
    },
    # Add more datasets here...
]


# ============================================
# SETUP OUTPUT DIRECTORIES
# ============================================
for split in ['train', 'val', 'test']:
    (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)


def remap_labels(label_path, class_map):
    """
    Read a YOLO label file and remap class IDs to unified IDs.
    Returns remapped lines, or empty list if no valid classes.
    """
    lines = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            old_id = int(parts[0])

            if class_map is None:
                # Already unified
                new_id = old_id
            else:
                # Map old_id -> class_name -> unified_id
                class_name = class_map.get(old_id)
                if class_name is None:
                    continue
                new_id = UNIFIED_CLASSES.get(class_name)
                if new_id is None:
                    continue

            parts[0] = str(new_id)
            lines.append(" ".join(parts))

    return lines


# ============================================
# MERGE LOOP
# ============================================
all_samples = []  # List of (image_path, label_lines)

for ds in DATASET_SOURCES:
    ds_path = Path(ds["path"])
    class_map = ds.get("class_map")

    if not ds_path.exists():
        print(f"⚠️  Dataset not found: {ds_path} — skipping")
        continue

    print(f"\n📂 Processing: {ds_path}")

    # Check all possible split folders
    for split_name in ['train', 'val', 'valid', 'test']:
        img_dir = ds_path / split_name / "images"
        lbl_dir = ds_path / split_name / "labels"

        if not img_dir.exists():
            continue

        images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))

        for img_path in images:
            # Find matching label
            label_path = lbl_dir / (img_path.stem + ".txt")
            if not label_path.exists():
                continue

            remapped = remap_labels(label_path, class_map)
            if remapped:
                all_samples.append((img_path, remapped))

    print(f"   Found {len(all_samples)} total samples so far")

# ============================================
# SHUFFLE AND SPLIT
# ============================================
random.seed(42)
random.shuffle(all_samples)

n = len(all_samples)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

splits = {
    'train': all_samples[:n_train],
    'val':   all_samples[n_train:n_train + n_val],
    'test':  all_samples[n_train + n_val:],
}

# ============================================
# COPY FILES
# ============================================
total = 0
for split_name, samples in splits.items():
    for i, (img_path, label_lines) in enumerate(samples):
        # Use unique name to avoid collisions across datasets
        new_name = f"{split_name}_{total + i:05d}"
        ext = img_path.suffix

        # Copy image
        dst_img = OUTPUT_DIR / split_name / "images" / f"{new_name}{ext}"
        shutil.copy(img_path, dst_img)

        # Write remapped label
        dst_lbl = OUTPUT_DIR / split_name / "labels" / f"{new_name}.txt"
        with open(dst_lbl, 'w') as f:
            f.write("\n".join(label_lines))

    total += len(samples)
    print(f"  {split_name}: {len(samples)} images")

# ============================================
# WRITE data.yaml
# ============================================
output_abs = str(OUTPUT_DIR.resolve())
yaml_content = f"""train: {output_abs}/train/images
val: {output_abs}/val/images
test: {output_abs}/test/images

nc: {len(UNIFIED_NAMES)}
names: {UNIFIED_NAMES}
"""

with open(OUTPUT_DIR / "data.yaml", "w") as f:
    f.write(yaml_content)

print(f"\n✅ Merged dataset ready at: {OUTPUT_DIR}")
print(f"   Total samples: {total}")
print(f"   Classes: {UNIFIED_NAMES}")
print(f"\n💡 Update your training cell:")
print(f'   DATA_YAML = str(Path("{OUTPUT_DIR}/data.yaml").resolve())')


📂 Processing: yolo_dataset
   Found 2495 total samples so far
⚠️  Dataset not found: extra_dataset_1 — skipping


KeyboardInterrupt: 

---
## Part 1: Dataset Preparation

### Option A: Using Roboflow (Easiest)
Uncomment and use this if you have a Roboflow dataset

In [1]:
# # Install Roboflow
# !pip install roboflow

# from roboflow import Roboflow

# # Initialize Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace().project("YOUR_PROJECT_NAME")
# dataset = project.version(1).download("yolov8")

# print(f"Dataset downloaded to: {dataset.location}")

In [7]:
import shutil
from os import path
import os
import json


# ============================================
# CONFIGURATION - UPDATE THESE PATHS
# ============================================
SRC_DIR = "./dentalai-DatasetNinja"  # YOUR SOURCE DATASET PATH
DEST_DIR = "./yolo_dataset"  # Output to working directory



# ============================================

# 2. CLASS DEFINITIONS (With Capitalization Fix)

# ============================================

# We map both 'Tooth' and 'tooth' to the same ID to be safe

classes = {

    # CLASS NAME      ID

    'tooth':          0,

    'Tooth':          0,

    

    'caries':         1,

    'Caries':         1,

    

    'cavity':         2,

    'Cavity':         2,

    

    'crack':          3,

    'Crack':          3

}



# ============================================

# 3. DIRECTORY SETUP

# ============================================

print(f"📍 Reading from: {SRC_DIR}")

print(f"📍 Writing to:   {DEST_DIR}")



for split in ['train', 'val', 'test']:

    os.makedirs(path.join(DEST_DIR, split, "images"), exist_ok=True)

    os.makedirs(path.join(DEST_DIR, split, "labels"), exist_ok=True)



# ============================================

# 4. CREATE DATA.YAML

# ============================================

# Note: 'nc' is 4 because we have ids 0, 1, 2, 3

yaml_content = f"""train: ../train/images

val: ../val/images

test: ../test/images



nc: 4

names: ['tooth', 'caries', 'cavity', 'crack']

"""



with open(path.join(DEST_DIR, "data.yaml"), "w") as f:

    f.write(yaml_content)



# ============================================

# 5. CONVERSION LOOP

# ============================================

# Map dataset folder names to YOLO folder names

# "valid" in source -> "val" in YOLO

dirs_map = {"train": "train", "valid": "val", "test": "test"}



total_converted = 0



for src_folder, dest_folder in dirs_map.items():

    src_img_path = path.join(SRC_DIR, src_folder, "img")

    src_ann_path = path.join(SRC_DIR, src_folder, "ann")

    

    if not path.exists(src_img_path):

        print(f"⚠️  Skipping '{src_folder}' - Folder not found at {src_img_path}")

        continue

        

    print(f"\nProcessing {src_folder}...")

    

    # Get list of JSON files

    json_files = [f for f in os.listdir(src_ann_path) if f.endswith('.json')] if path.exists(src_ann_path) else []

    

    for json_file in json_files:

        # 1. Load JSON

        with open(path.join(src_ann_path, json_file), 'r') as f:

            data = json.load(f)

            

        img_h = data['size']['height']

        img_w = data['size']['width']

        

        # 2. Prepare Label File Content

        yolo_lines = []

        

        for obj in data['objects']:

            label = obj['classTitle']

            

            # CHECK: Is this label in our allowed list?

            if label in classes:

                class_id = classes[label]

                

                # Get points

                points = obj['points']['exterior']

                xs = [p[0] for p in points]

                ys = [p[1] for p in points]

                

                # Bounding Box

                x_min, x_max = min(xs), max(xs)

                y_min, y_max = min(ys), max(ys)

                

                # YOLO Format (Normalized Center X, Center Y, Width, Height)

                bbox_w = x_max - x_min

                bbox_h = y_max - y_min

                x_center = x_min + (bbox_w / 2)

                y_center = y_min + (bbox_h / 2)

                

                # Normalize

                x_c_norm = x_center / img_w

                y_c_norm = y_center / img_h

                w_norm = bbox_w / img_w

                h_norm = bbox_h / img_h

                

                yolo_lines.append(f"{class_id} {x_c_norm} {y_c_norm} {w_norm} {h_norm}")

        

        # 3. Save Label File (Only if we found valid objects)

        if yolo_lines:

            txt_filename = json_file.replace('.json', '.txt').replace('.jpg', '') 

            # Note: often json is named "image.jpg.json", we want "image.txt"

            if not txt_filename.endswith('.txt'): txt_filename += '.txt'

            

            with open(path.join(DEST_DIR, dest_folder, "labels", txt_filename), "w") as f:

                f.write("\n".join(yolo_lines))

                

            # 4. Copy Image

            # Try to find the matching image (could be .jpg, .png, etc)

            base_name = json_file.replace('.json', '')

            image_src = path.join(src_img_path, base_name)

            

            # Simple copy

            if path.exists(image_src):

                shutil.copy(image_src, path.join(DEST_DIR, dest_folder, "images", base_name))

                total_converted += 1



print(f"\n✅ SUCCESS! Converted {total_converted} images.")

print(f"📂 Dataset is ready at: {DEST_DIR}")

📍 Reading from: ./dentalai-DatasetNinja
📍 Writing to:   ./yolo_dataset

Processing train...

Processing valid...

Processing test...

✅ SUCCESS! Converted 2495 images.
📂 Dataset is ready at: ./yolo_dataset




Here's the fixed notebook. I'll address all the major issues:

## Key Fixes Applied:
1. **Class mismatch** — Aligned `DentalDetector` with actual trained classes (`tooth` instead of individual tooth types, no `decay`)
2. **Label filename handling** — Robust parsing using `pathlib`
3. **`data.yaml` uses absolute paths**
4. **Epochs set to 100** with a reasonable default
5. **GPU auto-detection**
6. **Removed `model.save()`** — uses `best.pt` from training output
7. **W&B wrapped in try/finally**
8. **Per-class metrics actually printed**
9. **Flask API uses `tempfile`** for thread safety
10. **Path consistency** across all cells
11. **`DentalDetector` categorization** matches trained model classes



In [1]:
# Cell: Option B - Manual Dataset Conversion

import shutil
from os import path
from pathlib import Path
import os
import json


# ============================================
# CONFIGURATION - UPDATE THESE PATHS
# ============================================
SRC_DIR = "./dentalai-DatasetNinja"  # YOUR SOURCE DATASET PATH
DEST_DIR = "./yolo_dataset"  # Output to working directory

# Convert to absolute path for data.yaml compatibility
DEST_DIR_ABS = str(Path(DEST_DIR).resolve())

# ============================================
# 2. CLASS DEFINITIONS (With Capitalization Fix)
# ============================================
# We map both 'Tooth' and 'tooth' to the same ID to be safe
classes = {
    # CLASS NAME      ID
    'tooth':          0,
    'Tooth':          0,

    'caries':         1,
    'Caries':         1,

    'cavity':         2,
    'Cavity':         2,

    'crack':          3,
    'Crack':          3
}

CLASS_NAMES = ['tooth', 'caries', 'cavity', 'crack']

# ============================================
# 3. DIRECTORY SETUP
# ============================================
print(f"📍 Reading from: {SRC_DIR}")
print(f"📍 Writing to:   {DEST_DIR}")

for split in ['train', 'val', 'test']:
    os.makedirs(path.join(DEST_DIR, split, "images"), exist_ok=True)
    os.makedirs(path.join(DEST_DIR, split, "labels"), exist_ok=True)

# ============================================
# 4. CREATE DATA.YAML (using absolute paths)
# ============================================
yaml_content = f"""train: {DEST_DIR_ABS}/train/images
val: {DEST_DIR_ABS}/val/images
test: {DEST_DIR_ABS}/test/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""

with open(path.join(DEST_DIR, "data.yaml"), "w") as f:
    f.write(yaml_content)

print(f"📄 data.yaml written with absolute paths")

# ============================================
# 5. CONVERSION LOOP
# ============================================
dirs_map = {"train": "train", "valid": "val", "test": "test"}

total_converted = 0
skipped_labels = set()

for src_folder, dest_folder in dirs_map.items():
    src_img_path = path.join(SRC_DIR, src_folder, "img")
    src_ann_path = path.join(SRC_DIR, src_folder, "ann")

    if not path.exists(src_img_path):
        print(f"⚠️  Skipping '{src_folder}' - Folder not found at {src_img_path}")
        continue

    print(f"\nProcessing {src_folder}...")

    json_files = [f for f in os.listdir(src_ann_path) if f.endswith('.json')] if path.exists(src_ann_path) else []

    for json_file in json_files:
        # 1. Load JSON
        with open(path.join(src_ann_path, json_file), 'r') as f:
            data = json.load(f)

        img_h = data['size']['height']
        img_w = data['size']['width']

        # 2. Prepare Label File Content
        yolo_lines = []

        for obj in data['objects']:
            label = obj['classTitle']

            if label in classes:
                class_id = classes[label]

                points = obj['points']['exterior']
                xs = [p[0] for p in points]
                ys = [p[1] for p in points]

                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)

                # Clamp to image bounds
                x_min = max(0, x_min)
                y_min = max(0, y_min)
                x_max = min(img_w, x_max)
                y_max = min(img_h, y_max)

                bbox_w = x_max - x_min
                bbox_h = y_max - y_min

                # Skip degenerate boxes
                if bbox_w <= 0 or bbox_h <= 0:
                    continue

                x_center = x_min + (bbox_w / 2)
                y_center = y_min + (bbox_h / 2)

                x_c_norm = x_center / img_w
                y_c_norm = y_center / img_h
                w_norm = bbox_w / img_w
                h_norm = bbox_h / img_h

                yolo_lines.append(f"{class_id} {x_c_norm:.6f} {y_c_norm:.6f} {w_norm:.6f} {h_norm:.6f}")
            else:
                skipped_labels.add(label)

        # 3. Save Label File
        if yolo_lines:
            # Robust filename handling:
            # JSON is typically "image.jpg.json" -> we want label "image.txt"
            # Strip .json first, then strip the image extension to get the stem
            p = Path(json_file)
            stem = p.stem  # e.g. "image.jpg" from "image.jpg.json"
            image_filename = stem  # This is also the image filename
            label_stem = Path(stem).stem  # e.g. "image" from "image.jpg"
            txt_filename = label_stem + ".txt"

            with open(path.join(DEST_DIR, dest_folder, "labels", txt_filename), "w") as f:
                f.write("\n".join(yolo_lines))

            # 4. Copy Image
            image_src = path.join(src_img_path, image_filename)

            if path.exists(image_src):
                shutil.copy(image_src, path.join(DEST_DIR, dest_folder, "images", image_filename))
                total_converted += 1
            else:
                print(f"  ⚠️  Image not found: {image_src}")

if skipped_labels:
    print(f"\n⚠️  Skipped unknown labels: {skipped_labels}")

print(f"\n✅ SUCCESS! Converted {total_converted} images.")
print(f"📂 Dataset is ready at: {DEST_DIR}")

📍 Reading from: ./dentalai-DatasetNinja
📍 Writing to:   ./yolo_dataset
📄 data.yaml written with absolute paths

Processing train...

Processing valid...

Processing test...

✅ SUCCESS! Converted 2495 images.
📂 Dataset is ready at: ./yolo_dataset


In [ ]:
# Cell: Option C - If Your Data is Already in YOLO Format

# If your data is already in YOLO format, just set the path
DEST_DIR = "./kaggle/input/dentalai-dataset"
DEST_DIR_ABS = str(Path(DEST_DIR).resolve())

# Verify data.yaml exists
import yaml
with open(f"{DEST_DIR}/data.yaml", 'r') as f:
    config = yaml.safe_load(f)
    print("Dataset configuration:")
    print(f"  Classes ({config['nc']}): {config['names']}")
    print(f"  Train: {config['train']}")
    print(f"  Val: {config['val']}")

In [2]:
# Cell: Part 3 - Train the Model

from ultralytics import YOLO
from pathlib import Path
import os
import torch

# ============================================
# CONFIGURATION
# ============================================

# Path to your data.yaml file (use absolute path)
DATA_YAML = str(Path("yolo_dataset/data.yaml").resolve())

# Auto-detect device
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")
if DEVICE == "cpu":
    print("   ⚠️  Training on CPU will be very slow. GPU recommended.")

# Training hyperparameters
CONFIG = {
    "model": "yolov8s.pt",
    "epochs": 100,           # Reasonable default (increase for better results)
    "batch_size": 16,        # Reduce to 8 or 4 if out-of-memory
    "img_size": 640,
    "patience": 20,          # Early stopping patience
    "lr0": 0.001,
    "device": DEVICE,
}

# ============================================
# OPTIONAL: Weights & Biases Tracking
# ============================================
USE_WANDB = False  # Set to True if you want W&B logging

if USE_WANDB:
    try:
        import wandb
        wandb.init(
            project="Dental_Detection",
            name="YOLOv8_Tooth_and_Damage",
            config=CONFIG
        )
    except ImportError:
        print("⚠️  wandb not installed, skipping experiment tracking")
        USE_WANDB = False

# ============================================
# LOAD PRE-TRAINED MODEL
# ============================================
print(f"Loading pre-trained model: {CONFIG['model']}")
model = YOLO(CONFIG["model"])

print(f"\nModel info:")
print(f"  Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# ============================================
# TRAIN THE MODEL
# ============================================
print(f"\n{'='*60}")
print("STARTING TRAINING")
print(f"{'='*60}\n")

try:
    results = model.train(
        data=DATA_YAML,
        epochs=CONFIG["epochs"],
        batch=CONFIG["batch_size"],
        imgsz=CONFIG["img_size"],
        patience=CONFIG["patience"],

        project="Dental_Detection",
        name="YOLOv8_Training",
        exist_ok=True,
        save_period=10,

        # Data augmentation
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.1,
        scale=0.3,
        flipud=0.0,       # Dental X-rays: vertical flip usually harmful
        fliplr=0.5,
        mosaic=0.8,
        mixup=0.1,

        optimizer='AdamW',
        lr0=CONFIG["lr0"],
        weight_decay=0.0005,
        device=CONFIG["device"],
        val=True,
    )

    print(f"\n{'='*60}")
    print("TRAINING COMPLETED!")
    print(f"{'='*60}\n")

    # ============================================
    # BEST MODEL PATH (use this everywhere)
    # ============================================
    BEST_MODEL_PATH = str(Path("Dental_Detection/YOLOv8_Training/weights/best.pt"))
    MODEL_PATH = BEST_MODEL_PATH  # Global reference for later cells

    if Path(BEST_MODEL_PATH).exists():
        print(f"✅ Best model saved at: {BEST_MODEL_PATH}")
    else:
        # Fallback to last.pt
        LAST_MODEL_PATH = str(Path("Dental_Detection/YOLOv8_Training/weights/last.pt"))
        MODEL_PATH = LAST_MODEL_PATH
        print(f"✅ Model saved at: {LAST_MODEL_PATH}")

    # ============================================
    # PRINT FINAL METRICS
    # ============================================
    print("\nFinal Training Results:")
    print(f"  Best mAP50:    {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
    print(f"  Best mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

finally:
    if USE_WANDB:
        try:
            wandb.finish()
        except:
            pass

🖥️  Using device: CPU
   ⚠️  Training on CPU will be very slow. GPU recommended.
Loading pre-trained model: yolov8s.pt

Model info:
  Parameters: 11,166,560

STARTING TRAINING

New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.12  Python-3.14.2 torch-2.10.0+cpu CPU (AMD Ryzen 7 5700G with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\PerezKylerLee(Studen\SmileGuard-Train\yolo_dataset\data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v

KeyboardInterrupt: 

In [ ]:
# Cell: Part 4 - DentalDetector Class

from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import numpy as np

class DentalDetector:
    """
    Dental Detection System
    
    Detects:
    - Teeth (generic tooth class)
    - Damage: caries, cavity, crack
    - Position: upper/lower, left/right (heuristic from bbox)
    """
    
    def __init__(self, model_path):
        """
        Initialize the detector
        
        Args:
            model_path: Path to trained YOLOv8 model (.pt file)
        """
        print(f"Loading model from: {model_path}")
        self.model = YOLO(model_path)
        
        # Read class names from the model itself
        self.class_names = self.model.names  # dict: {0: 'tooth', 1: 'caries', ...}
        
        # Categorize classes based on what the model actually knows
        self.tooth_classes = []
        self.damage_classes = []
        
        known_damage = {'caries', 'cavity', 'crack', 'decay'}
        known_teeth = {'tooth', 'incisor', 'canine', 'premolar', 'molar'}
        
        for class_id, class_name in self.class_names.items():
            lower_name = class_name.lower()
            if lower_name in known_teeth:
                self.tooth_classes.append(class_name)
            elif lower_name in known_damage:
                self.damage_classes.append(class_name)
            else:
                print(f"  ⚠️  Unknown class '{class_name}' (id={class_id}) — treating as damage")
                self.damage_classes.append(class_name)
        
        print("✅ Model loaded successfully!")
        print(f"   Model classes: {list(self.class_names.values())}")
        print(f"   Tooth classes: {self.tooth_classes}")
        print(f"   Damage classes: {self.damage_classes}")
    
    def determine_position(self, bbox, img_height):
        """
        Determine if detection is in upper or lower jaw.
        
        Note: This is a simple heuristic based on image midpoint.
        It works best with standardized dental X-ray framing.
        For production use, consider a separate jaw segmentation model.
        """
        y_center = (bbox[1] + bbox[3]) / 2
        return 'upper' if y_center < img_height / 2 else 'lower'
    
    def determine_side(self, bbox, img_width):
        """Determine if detection is on left or right side."""
        x_center = (bbox[0] + bbox[2]) / 2
        return 'left' if x_center < img_width / 2 else 'right'
    
    def detect(self, image_path, conf_threshold=0.5, iou_threshold=0.45):
        """
        Perform detection on an image.
        
        Args:
            image_path: Path to the input image
            conf_threshold: Confidence threshold (0-1)
            iou_threshold: IoU threshold for NMS
            
        Returns:
            dict with 'teeth' and 'damages' detections
        """
        results = self.model(
            image_path,
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=False
        )
        
        img = Image.open(image_path)
        img_width, img_height = img.size
        
        teeth = []
        damages = []
        
        for result in results:
            boxes = result.boxes
            
            for box in boxes:
                class_id = int(box.cls.cpu().numpy())
                class_name = result.names[class_id]
                confidence = float(box.conf.cpu().numpy())
                bbox = box.xyxy[0].cpu().numpy().tolist()
                
                position = self.determine_position(bbox, img_height)
                side = self.determine_side(bbox, img_width)
                
                detection = {
                    'class': class_name,
                    'class_id': class_id,
                    'confidence': round(confidence, 4),
                    'bbox': [round(c, 2) for c in bbox],
                    'position': position,
                    'side': side,
                    'full_label': f"{position} {side} {class_name}"
                }
                
                if class_name in self.tooth_classes:
                    teeth.append(detection)
                else:
                    damages.append(detection)
        
        return {
            'teeth': teeth,
            'damages': damages,
            'image_path': str(image_path),
            'image_size': (img_width, img_height)
        }
    
    def visualize(self, image_path, detections, save_path=None):
        """Visualize detections on the image."""
        img = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(img)
        
        try:
            font = ImageFont.truetype("arial.ttf", 16)
        except:
            font = ImageFont.load_default()
        
        # Color map for different classes
        colors = {
            'tooth': "#00FF00",
            'caries': "#FF0000",
            'cavity': "#FF6600",
            'crack': "#FF00FF",
        }
        
        # Draw teeth (green boxes)
        for tooth in detections['teeth']:
            bbox = tooth['bbox']
            color = colors.get(tooth['class'], "#00FF00")
            
            draw.rectangle(bbox, outline=color, width=3)
            
            label = f"{tooth['full_label']} {tooth['confidence']:.0%}"
            text_bbox = draw.textbbox((bbox[0], bbox[1] - 25), label, font=font)
            draw.rectangle(text_bbox, fill=color)
            draw.text((bbox[0], bbox[1] - 25), label, fill="#000000", font=font)
        
        # Draw damages (red/orange boxes)
        for damage in detections['damages']:
            bbox = damage['bbox']
            color = colors.get(damage['class'], "#FF0000")
            
            draw.rectangle(bbox, outline=color, width=3)
            
            label = f"{damage['class'].upper()} {damage['confidence']:.0%}"
            text_bbox = draw.textbbox((bbox[0], bbox[1] - 25), label, font=font)
            draw.rectangle(text_bbox, fill=color)
            draw.text((bbox[0], bbox[1] - 25), label, fill="#FFFFFF", font=font)
        
        plt.figure(figsize=(15, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title(
            f"Detected: {len(detections['teeth'])} teeth, {len(detections['damages'])} damages",
            fontsize=16, fontweight='bold'
        )
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"✅ Visualization saved to: {save_path}")
        
        plt.show()
        return img
    
    def get_patient_report(self, detections):
        """Generate a human-readable patient report."""
        report = []
        report.append("=" * 60)
        report.append("           DENTAL DETECTION REPORT")
        report.append("=" * 60)
        report.append("")
        
        # Teeth summary
        report.append(f"🦷 TEETH DETECTED: {len(detections['teeth'])}")
        report.append("-" * 60)
        
        if not detections['teeth']:
            report.append("  No teeth detected in this image.")
        else:
            for i, tooth in enumerate(detections['teeth'], 1):
                report.append(
                    f"  {i}. {tooth['position'].capitalize()} {tooth['side']} "
                    f"{tooth['class']} (confidence: {tooth['confidence']:.1%})"
                )
        
        report.append("")
        report.append("=" * 60)
        
        # Damage summary
        report.append(f"⚠️  DAMAGE DETECTED: {len(detections['damages'])}")
        report.append("-" * 60)
        
        if not detections['damages']:
            report.append("  ✅ No damage detected! Teeth appear healthy.")
        else:
            for i, damage in enumerate(detections['damages'], 1):
                report.append(
                    f"  {i}. {damage['class'].upper()} on {damage['position']} {damage['side']} area"
                )
                report.append(f"     Confidence: {damage['confidence']:.1%}")
                report.append("")
        
        report.append("=" * 60)
        report.append("")
        report.append("NOTE: This is an AI-assisted screening tool.")
        report.append("Please consult a dental professional for diagnosis.")
        report.append("=" * 60)
        
        return "\n".join(report)
    
    def batch_detect(self, image_paths, conf_threshold=0.5):
        """Detect on multiple images."""
        results = []
        for i, img_path in enumerate(image_paths, 1):
            print(f"Processing [{i}/{len(image_paths)}]: {img_path}")
            detections = self.detect(img_path, conf_threshold)
            results.append(detections)
        return results

print("✅ DentalDetector class loaded successfully!")

In [ ]:
# Cell: Part 5 - Test the Model (Inference)

# ============================================
# LOAD THE TRAINED MODEL
# ============================================

# Use best.pt from training (set in Part 3), or override here:
if 'MODEL_PATH' not in dir() or not Path(MODEL_PATH).exists():
    MODEL_PATH = "Dental_Detection/YOLOv8_Training/weights/best.pt"

print(f"Using model: {MODEL_PATH}")
detector = DentalDetector(MODEL_PATH)

# ============================================
# TEST ON A SINGLE IMAGE
# ============================================

# Replace with your test image path
TEST_IMAGE = "yolo_dataset/test/images"  # UPDATE THIS to a specific image!

# Check if it's a directory — pick the first image
test_path = Path(TEST_IMAGE)
if test_path.is_dir():
    test_files = list(test_path.glob("*.jpg")) + list(test_path.glob("*.png"))
    if test_files:
        TEST_IMAGE = str(test_files[0])
        print(f"Auto-selected test image: {TEST_IMAGE}")
    else:
        raise FileNotFoundError(f"No images found in {TEST_IMAGE}")

# Perform detection
detections = detector.detect(
    TEST_IMAGE,
    conf_threshold=0.5
)

# Print report
print(detector.get_patient_report(detections))

# Visualize results
detector.visualize(
    TEST_IMAGE,
    detections,
    save_path="detection_result.jpg"
)

In [ ]:
# Cell: Part 8 - Model Evaluation

from ultralytics import YOLO
from pathlib import Path

# Load the trained model
model = YOLO(MODEL_PATH)

# Validate on test set
print("Evaluating model on test set...")
metrics = model.val(
    data=DATA_YAML,
    split='test',
    conf=0.5,
    iou=0.45
)

# Print overall metrics
print("\n" + "="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"\nInference speed: {metrics.speed['inference']:.1f}ms per image")

# Per-class metrics
print("\n" + "="*60)
print("PER-CLASS PERFORMANCE")
print("="*60)
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'mAP50':<12} {'mAP50-95':<12}")
print("-" * 63)

class_names = model.names
ap50_per_class = metrics.box.ap50       # shape: (num_classes,)
ap_per_class = metrics.box.ap           # shape: (num_classes,) — mean over IoU thresholds
p_per_class = metrics.box.p             # per-class precision
r_per_class = metrics.box.r             # per-class recall

for i, name in class_names.items():
    if i < len(ap50_per_class):
        print(
            f"{name:<15} "
            f"{p_per_class[i]:<12.4f} "
            f"{r_per_class[i]:<12.4f} "
            f"{ap50_per_class[i]:<12.4f} "
            f"{ap_per_class[i]:<12.4f}"
        )

In [ ]:
# Cell: Part 10 - Flask API for Integration

# Save this code to a separate file (e.g., app.py) and run it

flask_code = '''
from flask import Flask, request, jsonify
from PIL import Image
import io
import base64
import tempfile
import os

# Import DentalDetector (copy the class or import from a module)
from ultralytics import YOLO

app = Flask(__name__)

# Initialize detector
MODEL_PATH = "Dental_Detection/YOLOv8_Training/weights/best.pt"
model = YOLO(MODEL_PATH)

# Read class names from model
tooth_classes = set()
damage_classes = set()
known_damage = {"caries", "cavity", "crack", "decay"}
known_teeth = {"tooth", "incisor", "canine", "premolar", "molar"}

for cid, cname in model.names.items():
    if cname.lower() in known_teeth:
        tooth_classes.add(cname)
    else:
        damage_classes.add(cname)


@app.route("/detect", methods=["POST"])
def detect_dental():
    """
    API endpoint for dental detection.

    Expected input (JSON):
    {
        "image": "base64_encoded_image_string",
        "confidence": 0.5  // optional
    }
    """
    try:
        data = request.json
        image_data = base64.b64decode(data["image"])
        conf = data.get("confidence", 0.5)

        # Save to a unique temp file (thread-safe)
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            tmp_path = tmp.name
            img = Image.open(io.BytesIO(image_data))
            img.save(tmp_path)

        try:
            # Run detection
            results = model(tmp_path, conf=conf, iou=0.45, verbose=False)
            img_w, img_h = img.size

            teeth = []
            damages = []

            for result in results:
                for box in result.boxes:
                    class_id = int(box.cls.cpu().numpy())
                    class_name = result.names[class_id]
                    confidence = float(box.conf.cpu().numpy())
                    bbox = box.xyxy[0].cpu().numpy().tolist()

                    y_center = (bbox[1] + bbox[3]) / 2
                    x_center = (bbox[0] + bbox[2]) / 2
                    position = "upper" if y_center < img_h / 2 else "lower"
                    side = "left" if x_center < img_w / 2 else "right"

                    det = {
                        "class": class_name,
                        "confidence": round(confidence, 4),
                        "bbox": [round(c, 2) for c in bbox],
                        "position": position,
                        "side": side,
                    }

                    if class_name in tooth_classes:
                        teeth.append(det)
                    else:
                        damages.append(det)

            return jsonify({
                "success": True,
                "teeth": teeth,
                "damages": damages,
                "summary": {
                    "total_teeth": len(teeth),
                    "total_damages": len(damages),
                    "has_damage": len(damages) > 0,
                },
            })

        finally:
            os.unlink(tmp_path)

    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500


@app.route("/health", methods=["GET"])
def health_check():
    return jsonify({"status": "healthy", "model": "loaded", "classes": list(model.names.values())})


if __name__ == "__main__":
    app.run(debug=True, host="0.0.0.0", port=5000)
'''

# Write the Flask app to a file
with open("app.py", "w") as f:
    f.write(flask_code)

print("✅ Flask API saved to: app.py")
print("   Run with: python app.py")
print("   API will be available at: http://localhost:5000")
print("   Endpoints:")
print("     POST /detect  - Detect teeth and damage in an image")
print("     GET  /health  - Health check")















## Summary of All Changes

| Issue | Fix |
|---|---|
| Class mismatch (tooth types vs generic `tooth`) | `DentalDetector` now reads classes from the model itself, auto-categorizes |
| `decay` class referenced but never trained | Removed hardcoded expectation; uses dynamic class discovery |
| Fragile label filename parsing | Uses `pathlib.Path` for robust stem extraction |
| Relative paths in `data.yaml` | Uses absolute paths via `Path.resolve()` |
| `model.save()` not standard | Uses `best.pt` from training output directory |
| 1 epoch default | Changed to 100 with early stopping (`patience=20`) |
| Hardcoded CPU device | Auto-detects GPU with `torch.cuda.is_available()` |
| W&B left hanging on failure | Wrapped in `try/finally`, made optional (`USE_WANDB` flag) |
| Per-class metrics never printed | Extracts and prints `ap50`, `p`, `r` per class from `metrics.box` |
| Flask race condition on temp file | Uses `tempfile.NamedTemporaryFile` with `finally: os.unlink()` |
| `flipud=0.5` (bad for dental) | Set to `0.0` — vertical flipping produces unrealistic dental images |
| `DATA_YAML` / `MODEL_PATH` undefined in later cells | Consistent variable propagation; fallback checks with `Path.exists()` |
| No bounds clamping on annotations | Added clamping and degenerate box filtering |

---
## Summary & Next Steps

### What This Notebook Does:
1. ✅ Prepares dental dataset in YOLO format
2. ✅ Trains YOLOv8 to detect tooth types (incisors, canines, molars, premolars)
3. ✅ Trains YOLOv8 to detect damage (caries, cavity, decay, crack)
4. ✅ Provides position detection (upper/lower, left/right)
5. ✅ Generates patient reports
6. ✅ Includes API integration example

### Your Checklist:
- [ ] Prepare dataset with tooth type annotations
- [ ] Run Part 1 (dataset preparation)
- [ ] Run Part 3 (model training - 1-3 hours)
- [ ] Run Part 5 (test inference)
- [ ] Run Part 8 (evaluate performance)
- [ ] Integrate into your application using Part 10

### Tips for Better Results:
1. **More data is better**: Aim for 1000+ images
2. **Diverse images**: Different lighting, angles, cameras
3. **Accurate annotations**: Double-check your labels
4. **Higher epochs**: Try 150-200 epochs for better accuracy
5. **Larger model**: Use YOLOv8l or YOLOv8x if accuracy is critical

### Troubleshooting:
- **Low accuracy**: Need more training data or longer training
- **False positives**: Increase confidence threshold to 0.6-0.7
- **Out of memory**: Reduce batch size to 8 or 4
- **Slow training**: Use smaller model (yolov8n) or reduce image size

### Resources:
- Dataset labeling: https://roboflow.com or https://labelstud.io
- YOLOv8 docs: https://docs.ultralytics.com
- Dental datasets: Search Roboflow Universe, Kaggle

---

**Good luck with your dental detection project! 🦷**